# 🧠 PyTorch Geometric: From First Principles to Production GNNs

A progressive, self-contained notebook covering:

| Level | What You'll Build |
|-------|-------------------|
| 🟢 **Kindergarten** | `Data` objects, COO edge format, graph properties |
| 🟡 **Grade 5** | Edge-conditioned NNConv, batched node classification |
| 🟠 **Grade 10** | Edge-aware multi-head attention, residuals, AMP |
| 🔴 **Advanced** | Real datasets (Planetoid / OGB), neighbor sampling |
| 🟣 **Production** | Embeddings, PCA visualization, model checkpointing |
| ⚫ **Research** | Contrastive self-supervised pretraining (NT-Xent) |
| 🖥️ **Infra** | GPU monitoring dashboard, post-run analysis |

---

### Key Concepts Quick-Reference

```
Graph G = (V, E)
  V = nodes (vertices)  — represented as a feature matrix  x ∈ ℝ^{N×F}
  E = edges             — represented in COO format:  edge_index ∈ ℤ^{2×E}
                          edge_index[0] = source nodes
                          edge_index[1] = target nodes
  Optional: edge_attr ∈ ℝ^{E×D}  (edge features)
            y ∈ ℤ^N or ℤ^1      (node or graph labels)
```

**Message Passing** (the core GNN operation):
```
h_v^{(k)} = UPDATE( h_v^{(k-1)},  AGGREGATE( { MSG(h_v, h_u, e_uv) : u ∈ N(v) } ) )
```
Each layer: gather neighbor info → aggregate → update your own embedding.


## 0 · Installation
Run once per runtime. Detects Colab vs local automatically.

In [ ]:
import sys, subprocess, importlib

IN_COLAB = 'google.colab' in sys.modules

def pip(*args):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', *args])

# PyTorch Geometric core
if importlib.util.find_spec('torch_geometric') is None:
    pip('torch_geometric')

# Optional: OGB (Open Graph Benchmark)
if importlib.util.find_spec('ogb') is None:
    try:
        pip('ogb')
    except Exception:
        print("OGB install failed — OGB cells will be skipped.")

# Optional: pyg binary extensions (CUDA-accelerated scatter, sparse ops)
# Uncomment and set your torch+cuda version before running:
# TORCH_VER = "2.1.0"
# CUDA_VER  = "cu121"
# pip('pyg_lib', 'torch_scatter', 'torch_sparse', 'torch_cluster', 'torch_spline_conv',
#     '-f', f'https://data.pyg.org/whl/torch-{TORCH_VER}+{CUDA_VER}.html')

print("Environment ready ✓")


## 0b · PyTorch Serialization Safety (run before loading any OGB dataset)

PyTorch ≥ 2.6 requires explicit allowlisting of custom classes before `torch.load`.
PyG stores processed graph data as `.pt` files — run this cell **once** before loading
any OGB or cached PyG dataset, or you'll get a `WeightsOnlyLoadError`.


In [ ]:
import torch, importlib

def _allowlist_pyg_classes():
    """
    Allowlist all PyG Data-related classes that appear in serialized .pt files.
    Safe to call multiple times (add_safe_globals is idempotent).
    """
    safe = []
    # Core data containers
    for mod_path, names in [
        ('torch_geometric.data',         ['Data', 'HeteroData']),
        ('torch_geometric.data.data',    ['DataEdgeAttr', 'DataTensorAttr']),
        ('torch_geometric.data.storage', ['GlobalStorage']),
    ]:
        try:
            mod = importlib.import_module(mod_path)
            for name in names:
                if hasattr(mod, name):
                    safe.append(getattr(mod, name))
        except ImportError:
            pass

    if safe:
        torch.serialization.add_safe_globals(list(dict.fromkeys(safe)))
        print('Allowlisted:', [f'{c.__module__}.{c.__name__}' for c in safe])
    else:
        print('No PyG classes found to allowlist (unexpected).')

_allowlist_pyg_classes()


## 0c · Core Imports

In [ ]:
import os, time, random, math, warnings
from dataclasses import dataclass, field
from pathlib import Path
from typing import Optional, Tuple, Dict, List

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor
from torch.optim import Adam, AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import (
    NNConv, SAGEConv, GATConv,
    MessagePassing,
    global_mean_pool,
    BatchNorm,
)
from torch_geometric.utils import add_self_loops, softmax

warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')
print(f'PyTorch:      {torch.__version__}')
try:
    import torch_geometric
    print(f'PyG:          {torch_geometric.__version__}')
except Exception:
    pass

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed(42)


---
## 🟢 Part 1 · Kindergarten — The `Data` Object & Graph Fundamentals

### What is a graph in PyG?

PyG wraps every graph as a `torch_geometric.data.Data` object with named attributes:

| Attribute | Shape | Meaning |
|-----------|-------|---------|
| `x` | `[N, F]` | Node feature matrix |
| `edge_index` | `[2, E]` | COO-format directed edges |
| `edge_attr` | `[E, D]` | Edge feature matrix |
| `y` | `[N]` or `[1]` | Node or graph labels |

**COO = Coordinate format**: just two lists, `src[i] → dst[i]`.
For an undirected edge `(u, v)` you add both `(u,v)` and `(v,u)`.


In [ ]:
# ─── A directed 4-node cycle ────────────────────────────────────────────────
x = torch.tensor([
    [1.0, 0.5],   # node 0
    [0.2, 1.3],   # node 1
    [0.9, 0.1],   # node 2
    [0.4, 0.7],   # node 3
], dtype=torch.float)

edge_index = torch.tensor([
    [0, 1, 2, 3],   # sources  (COO row 0)
    [1, 2, 3, 0],   # targets  (COO row 1)  ← 0→1→2→3→0
], dtype=torch.long)

edge_attr = torch.tensor([[0.5], [0.3], [0.8], [0.2]])  # 1D edge feature
y         = torch.tensor([0, 1, 0, 1])                   # node labels (binary)

data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y)

print(data)
print()
print(f'  num_nodes    : {data.num_nodes}')
print(f'  num_edges    : {data.num_edges}')
print(f'  num_features : {data.num_features}')
print(f'  is_undirected: {data.is_undirected()}')
print(f'  has_self_loops: {data.has_self_loops()}')
print()

# ─── Validate the graph ─────────────────────────────────────────────────────
# edge_index should have non-negative indices within range
assert data.edge_index.min() >= 0
assert data.edge_index.max() < data.num_nodes
print('Edge index validation passed ✓')


### Visualising the graph

A simple networkx + matplotlib draw to see what we've built.


In [ ]:
try:
    import networkx as nx

    G = nx.DiGraph()
    G.add_nodes_from(range(data.num_nodes))
    edges = data.edge_index.t().tolist()
    edge_weights = data.edge_attr.squeeze().tolist()
    for (s, d), w in zip(edges, edge_weights):
        G.add_edge(s, d, weight=round(w, 2))

    pos = nx.circular_layout(G)
    node_colors = ['#4CAF50' if l == 0 else '#2196F3' for l in data.y.tolist()]
    edge_labels = nx.get_edge_attributes(G, 'weight')

    fig, ax = plt.subplots(figsize=(5, 4))
    nx.draw_networkx(G, pos, ax=ax,
                     node_color=node_colors, node_size=800,
                     font_color='white', font_weight='bold',
                     arrows=True, arrowsize=20, width=2)
    nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, ax=ax, font_size=9)
    ax.set_title('4-node directed cycle\nGreen=label 0, Blue=label 1', fontsize=11)
    plt.tight_layout(); plt.show()
except ImportError:
    print('networkx not installed — skipping visualisation.')
    print('Install with: pip install networkx')


---
## 🟡 Part 2 · Grade 5 — Edge-Conditioned Node Classification with NNConv

### Why edge features matter

Standard GCN/GraphSAGE *ignore* edge attributes.  
**NNConv** (edge-conditioned convolution) instead uses a small neural network
to map each edge's feature vector to a per-edge weight matrix, so the messages
become: `M_{u→v} = edge_net(e_{uv}) · h_u`.

This is critical when edge features carry semantic meaning (bond type in
molecules, relationship strength in social graphs, road capacity in routing).

### Architecture overview
```
Input ─► Linear ─► ReLU
              │
              ▼
        NNConv (edge-net ─► weight matrix ─► transforms message)
              ▼
         BatchNorm ─► ReLU
              │
        NNConv  (second layer)
              ▼
         BatchNorm ─► ReLU
              │
            MLP ─► class logits  (one per node)
```


In [ ]:
# ─── Synthetic dataset: many small random cycle graphs ──────────────────────

def make_cycle_graph(n_nodes: int = 8, node_feat_dim: int = 4,
                     edge_feat_dim: int = 2, num_classes: int = 3,
                     seed: Optional[int] = None) -> Data:
    """
    Directed cycle 0→1→…→(n-1)→0 with random node/edge features.
    Labels: sign of the first node feature (makes learning possible).
    """
    if seed is not None:
        torch.manual_seed(seed)
    x         = torch.randn((n_nodes, node_feat_dim))
    src       = torch.arange(n_nodes)
    dst       = torch.roll(src, shifts=-1)
    edge_index = torch.stack([src, dst])
    edge_attr  = torch.randn((n_nodes, edge_feat_dim))
    # Learnable label: bucket the first feature into num_classes equal bins
    y = torch.bucketize(x[:, 0], torch.linspace(x[:, 0].min(), x[:, 0].max(), num_classes - 1))
    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y)


def create_dataset(num_graphs: int = 400, min_nodes: int = 6, max_nodes: int = 14,
                   node_feat_dim: int = 4, edge_feat_dim: int = 2,
                   num_classes: int = 3) -> List[Data]:
    dataset = []
    for i in range(num_graphs):
        n = torch.randint(min_nodes, max_nodes + 1, (1,)).item()
        dataset.append(make_cycle_graph(n, node_feat_dim, edge_feat_dim, num_classes, seed=i))
    return dataset


# Quick sanity check
sample = make_cycle_graph(seed=0)
print(sample)
print(f'  Label distribution: {sample.y.bincount().tolist()}')


In [ ]:
# ─── NNConv-based NodeClassifier ────────────────────────────────────────────

def make_edge_net(edge_dim: int, in_ch: int, out_ch: int, hidden: int = 64) -> nn.Sequential:
    """
    Maps edge features (ℝ^edge_dim) → weight matrix (ℝ^{in_ch × out_ch}).
    This tiny MLP is the 'conditioning network' of NNConv.
    """
    return nn.Sequential(
        nn.Linear(edge_dim, hidden), nn.ReLU(),
        nn.Linear(hidden, in_ch * out_ch),
    )


class NodeClassifier(nn.Module):
    """
    Two-layer NNConv model for node-level classification.

    Args:
        in_channels:     input node feature dimension
        edge_dim:        edge feature dimension
        hidden_channels: internal representation width
        num_classes:     number of output classes
        dropout:         dropout probability before final MLP
    """
    def __init__(self, in_channels: int, edge_dim: int,
                 hidden_channels: int = 64, num_classes: int = 3,
                 dropout: float = 0.4):
        super().__init__()
        H = hidden_channels

        self.lin_in   = nn.Linear(in_channels, H)

        self.edge_net1 = make_edge_net(edge_dim, H, H)
        self.conv1     = NNConv(H, H, self.edge_net1, aggr='mean')
        self.bn1       = BatchNorm(H)

        self.edge_net2 = make_edge_net(edge_dim, H, H)
        self.conv2     = NNConv(H, H, self.edge_net2, aggr='mean')
        self.bn2       = BatchNorm(H)

        self.mlp = nn.Sequential(
            nn.Linear(H, H), nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(H, num_classes),
        )

    def forward(self, x: Tensor, edge_index: Tensor, edge_attr: Tensor) -> Tensor:
        x = F.relu(self.lin_in(x))
        x = F.relu(self.bn1(self.conv1(x, edge_index, edge_attr)))
        x = F.relu(self.bn2(self.conv2(x, edge_index, edge_attr)))
        return self.mlp(x)           # [N, num_classes]


# Count parameters
model_g5 = NodeClassifier(in_channels=4, edge_dim=2, hidden_channels=64, num_classes=3)
n_params  = sum(p.numel() for p in model_g5.parameters() if p.requires_grad)
print(f'NodeClassifier  params: {n_params:,}')
print(model_g5)


In [ ]:
# ─── Training utilities ──────────────────────────────────────────────────────

def train_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = total_correct = total_nodes = 0
    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        logits = model(batch.x, batch.edge_index, batch.edge_attr)
        loss   = F.cross_entropy(logits, batch.y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)
        optimizer.step()
        total_loss    += loss.item() * batch.num_nodes
        total_correct += (logits.argmax(-1) == batch.y).sum().item()
        total_nodes   += batch.num_nodes
    return total_loss / total_nodes, total_correct / total_nodes


@torch.no_grad()
def eval_epoch(model, loader, device):
    model.eval()
    total_loss = total_correct = total_nodes = 0
    for batch in loader:
        batch = batch.to(device)
        logits = model(batch.x, batch.edge_index, batch.edge_attr)
        loss   = F.cross_entropy(logits, batch.y)
        total_loss    += loss.item() * batch.num_nodes
        total_correct += (logits.argmax(-1) == batch.y).sum().item()
        total_nodes   += batch.num_nodes
    return total_loss / total_nodes, total_correct / total_nodes


# ─── Run training ───────────────────────────────────────────────────────────
set_seed(42)
dataset = create_dataset(400, node_feat_dim=4, edge_feat_dim=2, num_classes=3)
train_ds, val_ds, test_ds = dataset[:320], dataset[320:360], dataset[360:]

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=32)
test_loader  = DataLoader(test_ds,  batch_size=32)

model_g5  = NodeClassifier(4, 2, 64, 3, dropout=0.4).to(DEVICE)
optimizer = Adam(model_g5.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = CosineAnnealingLR(optimizer, T_max=30)

best_val_acc = 0.0
best_state   = None
history      = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

for epoch in range(1, 31):
    tr_loss, tr_acc = train_epoch(model_g5, train_loader, optimizer, DEVICE)
    va_loss, va_acc = eval_epoch(model_g5, val_loader,   DEVICE)
    scheduler.step()
    for k, v in zip(history, [tr_loss, va_loss, tr_acc, va_acc]):
        history[k].append(v)
    if va_acc > best_val_acc:
        best_val_acc = va_acc
        best_state   = {k: v.cpu().clone() for k, v in model_g5.state_dict().items()}
    if epoch % 5 == 0 or epoch == 1:
        print(f'Epoch {epoch:02d}  train {tr_loss:.4f}/{tr_acc:.3f}  val {va_loss:.4f}/{va_acc:.3f}')

model_g5.load_state_dict(best_state)
te_loss, te_acc = eval_epoch(model_g5, test_loader, DEVICE)
print(f'\nTest  loss={te_loss:.4f}  acc={te_acc:.3f}  (best val acc={best_val_acc:.3f})')


In [ ]:
# ─── Training curves ─────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.5))
epochs_x = range(1, len(history['train_loss']) + 1)

ax1.plot(epochs_x, history['train_loss'], label='Train'); ax1.plot(epochs_x, history['val_loss'], label='Val')
ax1.set_title('Loss'); ax1.set_xlabel('Epoch'); ax1.legend(); ax1.grid(True, alpha=0.4)

ax2.plot(epochs_x, history['train_acc'], label='Train'); ax2.plot(epochs_x, history['val_acc'], label='Val')
ax2.set_title('Accuracy'); ax2.set_xlabel('Epoch'); ax2.legend(); ax2.grid(True, alpha=0.4)

plt.suptitle('Grade 5 — NNConv NodeClassifier', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()


---
## 🟠 Part 3 · Grade 10 — Edge-Aware Multi-Head Attention GNN

### Key upgrades over Grade 5

| Feature | Grade 5 | Grade 10 |
|---------|---------|----------|
| Edge conditioning | weight matrix (NNConv) | **attention score** |
| Normalization | BatchNorm | **LayerNorm** |
| Residual connections | ✗ | ✓ |
| Learning rate | fixed Adam | **cosine annealing** |
| Gradient control | clip | clip + **AMP** |
| Overfitting protection | Dropout | Dropout + **early stopping** |

### Edge-Aware GAT attention score

```
α_{uv} = LeakyReLU( a_src · h_u + a_dst · h_v + a_edge · e_{uv} )
```

`softmax` over incoming edges, then use `α` to weight-sum messages `h_u + e_{uv}`.


In [ ]:
# ─── EdgeAwareGAT: custom MessagePassing ────────────────────────────────────

class EdgeAwareGAT(MessagePassing):
    """
    Multi-head graph attention where attention scores depend on
    the source node, target node, AND edge features simultaneously.

    After propagation each head produces a hidden_channels/heads vector;
    concat=True concatenates all heads back to hidden_channels.
    """
    def __init__(self, in_channels: int, out_channels: int,
                 edge_dim: int, heads: int = 4, concat: bool = True):
        super().__init__(aggr='add')
        self.in_ch  = in_channels
        self.out_ch = out_channels
        self.heads  = heads
        self.concat = concat

        self.lin_node = nn.Linear(in_channels,  heads * out_channels, bias=False)
        self.lin_edge = nn.Linear(edge_dim,      heads * out_channels, bias=False)
        # Learnable attention vectors (one scalar per head per feature)
        self.a_src  = nn.Parameter(torch.empty(1, heads, out_channels))
        self.a_dst  = nn.Parameter(torch.empty(1, heads, out_channels))
        self.a_edge = nn.Parameter(torch.empty(1, heads, out_channels))
        self._reset_params()

    def _reset_params(self):
        for t in (self.lin_node.weight, self.lin_edge.weight,
                  self.a_src, self.a_dst, self.a_edge):
            nn.init.xavier_uniform_(t)

    def forward(self, x: Tensor, edge_index: Tensor, edge_attr: Tensor) -> Tensor:
        # Project nodes & edges to [N/E, heads, out_ch]
        x_h    = self.lin_node(x).view(-1, self.heads, self.out_ch)
        edge_h = self.lin_edge(edge_attr).view(-1, self.heads, self.out_ch)
        # Self-loops: nodes attend to themselves (adds identity bias)
        edge_index, edge_h = add_self_loops(
            edge_index, num_nodes=x.size(0), edge_attr=edge_h)
        return self.propagate(edge_index, x=x_h, edge_attr=edge_h)

    def message(self, x_j, x_i, edge_attr, index, ptr=None):
        # Attention logit: dot-product with learned vectors, summed across feature dim
        alpha = ((x_j  * self.a_src).sum(-1) +
                 (x_i  * self.a_dst).sum(-1) +
                 (edge_attr * self.a_edge).sum(-1))            # [E, heads]
        alpha = F.leaky_relu(alpha, negative_slope=0.2)
        alpha = softmax(alpha, index, ptr).unsqueeze(-1)       # [E, heads, 1]
        return alpha * (x_j + edge_attr)                       # [E, heads, out_ch]

    def update(self, aggr_out: Tensor) -> Tensor:
        # aggr_out: [N, heads, out_ch]
        return (aggr_out.view(-1, self.heads * self.out_ch)
                if self.concat else aggr_out.mean(1))


# ─── Deep GNN with residuals ─────────────────────────────────────────────────

class DeepEdgeGNN(nn.Module):
    """
    Stack of EdgeAwareGAT layers with:
    - Pre-norm residual connections (LayerNorm before sublayer)
    - Cosine-annealed training (handled outside)
    - Optional AMP-safe forward pass
    """
    def __init__(self, in_channels: int, edge_dim: int, hidden: int,
                 num_layers: int, num_classes: int,
                 heads: int = 4, dropout: float = 0.2):
        super().__init__()
        assert hidden % heads == 0, 'hidden must be divisible by heads'
        self.input_lin = nn.Linear(in_channels, hidden)
        self.layers = nn.ModuleList([
            EdgeAwareGAT(hidden, hidden // heads, edge_dim, heads=heads)
            for _ in range(num_layers)
        ])
        self.norms     = nn.ModuleList([nn.LayerNorm(hidden) for _ in range(num_layers)])
        self.dropout   = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden, num_classes)

    def forward(self, x: Tensor, edge_index: Tensor, edge_attr: Tensor) -> Tensor:
        x = F.relu(self.input_lin(x))
        for conv, ln in zip(self.layers, self.norms):
            h = conv(x, edge_index, edge_attr)
            x = F.relu(x + self.dropout(ln(h)))     # pre-norm residual
        return self.classifier(x)


model_g10 = DeepEdgeGNN(in_channels=4, edge_dim=2, hidden=64,
                         num_layers=4, num_classes=3, heads=4, dropout=0.2)
print(f'DeepEdgeGNN  params: {sum(p.numel() for p in model_g10.parameters()):,}')


In [ ]:
# ─── Training with AMP + early stopping ──────────────────────────────────────

@dataclass
class EarlyStop:
    patience: int = 10
    best_val: float = 0.0
    wait: int      = 0
    best_state: dict = field(default_factory=dict)

    def step(self, val_acc: float, model: nn.Module) -> bool:
        """Returns True if training should stop."""
        if val_acc > self.best_val + 1e-4:
            self.best_val   = val_acc
            self.wait       = 0
            self.best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            self.wait += 1
        return self.wait >= self.patience


def train_g10(model, train_loader, val_loader, device, epochs=60, patience=12):
    model.to(device)
    opt       = AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    sched     = CosineAnnealingLR(opt, T_max=epochs)
    scaler    = torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda'))
    stopper   = EarlyStop(patience=patience)
    history   = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

    for epoch in range(1, epochs + 1):
        # ── train ──
        model.train()
        t_loss = t_correct = t_nodes = 0
        for batch in train_loader:
            batch = batch.to(device)
            opt.zero_grad()
            with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
                out  = model(batch.x, batch.edge_index, batch.edge_attr)
                loss = F.cross_entropy(out, batch.y)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)
            scaler.step(opt); scaler.update()
            t_loss    += loss.item() * batch.num_nodes
            t_correct += (out.argmax(-1) == batch.y).sum().item()
            t_nodes   += batch.num_nodes
        sched.step()

        # ── validate ──
        with torch.no_grad():
            model.eval()
            v_loss = v_correct = v_nodes = 0
            for batch in val_loader:
                batch = batch.to(device)
                out   = model(batch.x, batch.edge_index, batch.edge_attr)
                v_loss    += F.cross_entropy(out, batch.y).item() * batch.num_nodes
                v_correct += (out.argmax(-1) == batch.y).sum().item()
                v_nodes   += batch.num_nodes

        tr_a = t_correct / t_nodes; va_a = v_correct / v_nodes
        for k, v in zip(history, [t_loss/t_nodes, v_loss/v_nodes, tr_a, va_a]):
            history[k].append(v)

        if epoch % 10 == 0 or epoch == 1:
            print(f'Epoch {epoch:03d}  '
                  f'train {t_loss/t_nodes:.4f}/{tr_a:.3f}  '
                  f'val {v_loss/v_nodes:.4f}/{va_a:.3f}  '
                  f'lr={sched.get_last_lr()[0]:.5f}')

        if stopper.step(va_a, model):
            print(f'Early stopping at epoch {epoch} (best val acc={stopper.best_val:.3f})')
            break

    model.load_state_dict(stopper.best_state)
    return model, history


set_seed(42)
dataset2      = create_dataset(400, node_feat_dim=4, edge_feat_dim=2, num_classes=3)
tr2, va2, te2 = dataset2[:320], dataset2[320:360], dataset2[360:]
tl2 = DataLoader(tr2, batch_size=16, shuffle=True)
vl2 = DataLoader(va2, batch_size=32)
teldr = DataLoader(te2, batch_size=32)

model_g10 = DeepEdgeGNN(4, 2, 64, 4, 3, heads=4, dropout=0.2)
model_g10, hist_g10 = train_g10(model_g10, tl2, vl2, DEVICE)
te_loss2, te_acc2   = eval_epoch(model_g10, teldr, DEVICE)
print(f'\nTest  loss={te_loss2:.4f}  acc={te_acc2:.3f}')


---
## 🔴 Part 4 · Advanced — Real Datasets (Planetoid / OGB)

### Planetoid datasets
Built into PyG: `Cora`, `CiteSeer`, `PubMed`.
Each is a **single large graph** (full-batch training) with train/val/test node masks.

### OGB datasets
`ogbn-arxiv`, `ogbn-products`, etc. — much larger, with canonical splits.
Labels stored as `y.shape = [N, 1]` → squeeze before use.

### Full-batch vs mini-batch
| | Full-batch | NeighborSampler (mini-batch) |
|---|---|---|
| Memory | entire graph on GPU | sampled subgraph only |
| Accuracy | generally better | stochastic, needs tuning |
| Scalability | limited by GPU VRAM | scales to billions of nodes |


In [ ]:
# ─── Split helpers (works for Planetoid + OGB) ───────────────────────────────

def make_bool_mask(indices, num_nodes: int) -> torch.Tensor:
    """Convert index tensor/list to boolean mask."""
    m = torch.zeros(num_nodes, dtype=torch.bool)
    m[torch.as_tensor(indices, dtype=torch.long).view(-1)] = True
    return m


def get_splits(data: Data, ds=None) -> Tuple[Tensor, Tensor, Tensor]:
    """
    Returns (train_mask, val_mask, test_mask) as boolean tensors.
    Tries: (1) data.{train,val,test}_mask  (2) ds.get_idx_split() (OGB).
    Raises RuntimeError if splits cannot be found.
    """
    N = data.num_nodes
    train_mask = getattr(data, 'train_mask', None)
    val_mask   = getattr(data, 'val_mask',   None)
    test_mask  = getattr(data, 'test_mask',  None)

    # Convert integer index tensors → bool
    for m in (train_mask, val_mask, test_mask):
        if m is not None and m.dtype != torch.bool:
            m = make_bool_mask(m, N)

    # OGB: get_idx_split
    if (train_mask is None or val_mask is None or test_mask is None) and ds is not None:
        try:
            splits = ds.get_idx_split()
            if train_mask is None: train_mask = make_bool_mask(splits['train'], N)
            if val_mask   is None: val_mask   = make_bool_mask(splits.get('valid', splits.get('val', None)), N)
            if test_mask  is None: test_mask  = make_bool_mask(splits['test'],  N)
        except Exception:
            pass

    if any(m is None for m in (train_mask, val_mask, test_mask)):
        avail = [k for k in dir(data) if 'mask' in k.lower() or 'idx' in k.lower()]
        raise RuntimeError(f'Cannot find splits. Available: {avail}')

    return train_mask, val_mask, test_mask


In [ ]:
# ─── Hybrid SAGE + Attention model ──────────────────────────────────────────

class HybridSAGEAttention(nn.Module):
    """
    GraphSAGE layers for robust inductive aggregation,
    topped with a GAT layer for expressive attention.

    GraphSAGE: h_v = W · CONCAT(h_v, MEAN({h_u : u∈N(v)}))
    GAT head : weighted attention over neighbors

    Why this combo?
    - SAGE generalises to unseen nodes (inductive)
    - GAT learns which neighbors are most relevant
    - Using SAGE early stabilises training; GAT refines at the end
    """
    def __init__(self, in_dim: int, hidden_dim: int, out_dim: int,
                 num_layers: int = 3, heads: int = 4, dropout: float = 0.2):
        super().__init__()
        assert num_layers >= 2, 'Need at least 2 layers (SAGE + GAT)'
        self.dropout   = dropout
        self.input_lin = nn.Linear(in_dim, hidden_dim)
        self.sage      = nn.ModuleList([SAGEConv(hidden_dim, hidden_dim)
                                        for _ in range(num_layers - 1)])
        self.attn      = GATConv(hidden_dim, hidden_dim // heads, heads=heads, concat=True)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2), nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, out_dim),
        )

    def forward(self, x: Tensor, edge_index: Tensor) -> Tensor:
        x = F.relu(self.input_lin(x))
        for sage_layer in self.sage:
            x = F.relu(sage_layer(x, edge_index))
            x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.elu(self.attn(x, edge_index))   # attention final layer
        return self.classifier(x)


In [ ]:
# ─── Full-batch training on Planetoid ────────────────────────────────────────

def train_planetoid(dataset_name: str = 'Cora', epochs: int = 200, patience: int = 20):
    from torch_geometric.datasets import Planetoid

    ds   = Planetoid(root='./data', name=dataset_name)
    data = ds[0]
    train_mask, val_mask, test_mask = get_splits(data, ds)
    data = data.to(DEVICE)
    train_mask = train_mask.to(DEVICE)
    val_mask   = val_mask.to(DEVICE)
    test_mask  = test_mask.to(DEVICE)

    in_dim  = data.num_node_features
    out_dim = int(data.y.max().item()) + 1
    model   = HybridSAGEAttention(in_dim, hidden_dim=128, out_dim=out_dim,
                                   num_layers=3, heads=4, dropout=0.3).to(DEVICE)
    opt     = AdamW(model.parameters(), lr=5e-3, weight_decay=5e-4)
    sched   = CosineAnnealingLR(opt, T_max=epochs)
    stopper = EarlyStop(patience=patience)
    history = []

    for epoch in range(1, epochs + 1):
        model.train(); opt.zero_grad()
        out  = model(data.x, data.edge_index)
        loss = F.cross_entropy(out[train_mask], data.y[train_mask])
        loss.backward(); opt.step(); sched.step()

        model.eval()
        with torch.no_grad():
            pred     = model(data.x, data.edge_index).argmax(-1)
            val_acc  = (pred[val_mask]  == data.y[val_mask]).float().mean().item()
            test_acc = (pred[test_mask] == data.y[test_mask]).float().mean().item()
        history.append({'epoch': epoch, 'loss': loss.item(),
                        'val_acc': val_acc, 'test_acc': test_acc})
        if epoch % 25 == 0:
            print(f'[{dataset_name}] Ep {epoch:03d}  loss {loss.item():.4f}  '
                  f'val {val_acc:.4f}  test {test_acc:.4f}')
        if stopper.step(val_acc, model):
            print(f'Early stop at epoch {epoch}')
            break

    model.load_state_dict(stopper.best_state)
    return model, history, {'dataset': dataset_name,
                            'best_val': stopper.best_val, 'final_test': test_acc}


# Run — downloads ~3 MB on first use
set_seed(42)
model_cora, hist_cora, metrics_cora = train_planetoid('Cora', epochs=200, patience=20)
print('\nMetrics:', metrics_cora)


---
## 🟣 Part 5 · Production — Embeddings & Visualisation

Node embeddings from the penultimate layer of a trained GNN can be:
- Visualised with PCA or t-SNE to inspect cluster quality
- Used as features for downstream tasks (transfer learning)
- Saved to disk for batch inference pipelines

The key insight: **a well-trained GNN's embeddings should form tight,
well-separated clusters per class** — even in 2D projections.


In [ ]:
# ─── Extract embeddings from Cora model ──────────────────────────────────────

from torch_geometric.datasets import Planetoid
from sklearn.decomposition import PCA

ds_cora   = Planetoid(root='./data', name='Cora')
data_cora = ds_cora[0].to(DEVICE)

# Patch HybridSAGEAttention to expose embeddings
class HybridSAGEAttentionEmb(HybridSAGEAttention):
    def embed(self, x: Tensor, edge_index: Tensor) -> Tensor:
        """Return the representation BEFORE the classification head."""
        x = F.relu(self.input_lin(x))
        for sage_layer in self.sage:
            x = F.relu(sage_layer(x, edge_index))
            x = F.dropout(x, p=self.dropout, training=False)
        x = F.elu(self.attn(x, edge_index))
        return x   # [N, hidden_dim]

# Re-use weights from trained model (architecture is identical)
emb_model = HybridSAGEAttentionEmb(
    data_cora.num_node_features, 128,
    int(data_cora.y.max().item()) + 1, 3, 4, 0.3
).to(DEVICE)
emb_model.load_state_dict(model_cora.state_dict())
emb_model.eval()

with torch.no_grad():
    embeddings = emb_model.embed(data_cora.x, data_cora.edge_index).cpu().numpy()
labels = data_cora.y.cpu().numpy()

print(f'Embeddings shape: {embeddings.shape}')

# ─── PCA 2D visualisation ───────────────────────────────────────────────────
pca   = PCA(n_components=2, random_state=42)
emb2d = pca.fit_transform(embeddings)
print(f'PCA explained variance: {pca.explained_variance_ratio_.sum()*100:.1f}%')

fig, ax = plt.subplots(figsize=(8, 6))
scatter = ax.scatter(emb2d[:, 0], emb2d[:, 1],
                     c=labels, cmap='tab10', s=18, alpha=0.75, linewidths=0)
plt.colorbar(scatter, ax=ax, label='Class')
ax.set_title('PCA of Cora node embeddings (HybridSAGEAttention)', fontsize=12)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
plt.tight_layout(); plt.show()

# Optional: save
out_dir = Path('./outputs'); out_dir.mkdir(exist_ok=True)
np.save(out_dir / 'cora_embeddings.npy', embeddings)
print('Saved embeddings to', out_dir / 'cora_embeddings.npy')


In [ ]:
# ─── t-SNE (slower but often more revealing) ─────────────────────────────────
try:
    from sklearn.manifold import TSNE
    tsne  = TSNE(n_components=2, perplexity=40, n_iter=1000, random_state=42, verbose=0)
    emb_t = tsne.fit_transform(embeddings)

    fig, ax = plt.subplots(figsize=(8, 6))
    sc = ax.scatter(emb_t[:, 0], emb_t[:, 1], c=labels, cmap='tab10', s=18, alpha=0.75, linewidths=0)
    plt.colorbar(sc, ax=ax, label='Class')
    ax.set_title('t-SNE of Cora node embeddings', fontsize=12)
    plt.tight_layout(); plt.show()
except ImportError:
    print('scikit-learn t-SNE not available.')


---
## ⚫ Part 6 · Research — Contrastive Self-Supervised Pretraining

### Motivation
Labels are expensive. Can we learn good node representations
**without any labels**, purely from the graph structure?

### GraphCL / GCC approach
1. Create **two augmented views** of the same (sub)graph
2. Train an encoder so the same node's views are **similar** (positive pair)
   while different nodes' views are **dissimilar** (negatives)
3. Use the **NT-Xent** loss (normalised temperature-scaled cross-entropy)

### Augmentations used here
| Augmentation | What it does |
|---|---|
| Feature masking | Randomly zero out node feature dimensions |
| Edge dropout | Randomly remove a fraction of edges |

After pretraining, the encoder's weights are a warm-start for fine-tuning
on a small labelled dataset — often beating random init significantly.


In [ ]:
# ─── Augmentation functions ──────────────────────────────────────────────────

def aug_feat_mask(x: Tensor, prob: float = 0.2) -> Tensor:
    """Randomly zero out each feature with probability `prob`."""
    return x * (torch.rand_like(x) > prob).to(x.dtype)


def aug_edge_dropout(edge_index: Tensor, prob: float = 0.2) -> Tensor:
    """Randomly drop edges with probability `prob`."""
    keep = torch.rand(edge_index.size(1), device=edge_index.device) > prob
    return edge_index[:, keep]


# ─── NT-Xent contrastive loss ────────────────────────────────────────────────

def nt_xent(z1: Tensor, z2: Tensor, temperature: float = 0.5) -> Tensor:
    """
    NT-Xent (normalised temperature-scaled cross-entropy) loss.

    Given embeddings z1 and z2 for B samples, the positive pair for
    sample i is (z1[i], z2[i]); all other 2B-2 pairs are negatives.

    This is the SimCLR loss applied at the node/graph level.
    """
    B = z1.size(0)
    z = torch.cat([F.normalize(z1, dim=1),
                   F.normalize(z2, dim=1)], dim=0)           # [2B, D]

    sim = torch.matmul(z, z.T) / temperature                 # [2B, 2B]
    # Remove self-similarity
    mask = ~torch.eye(2 * B, dtype=torch.bool, device=z.device)
    exp_sim = torch.exp(sim) * mask.float()

    # Positive similarities (i ↔ i+B)
    pos = torch.exp(
        (F.normalize(z1, dim=1) * F.normalize(z2, dim=1)).sum(1) / temperature
    )
    pos = torch.cat([pos, pos])                               # [2B]

    loss = -torch.log(pos / exp_sim.sum(1))
    return loss.mean()


In [ ]:
# ─── Mini-batch contrastive pretraining ──────────────────────────────────────

class SAGEPretrainEncoder(nn.Module):
    """Lightweight 2-layer SAGE encoder for self-supervised pretraining."""
    def __init__(self, in_dim: int, hidden: int = 128, emb_dim: int = 64):
        super().__init__()
        self.conv1   = SAGEConv(in_dim,  hidden)
        self.conv2   = SAGEConv(hidden,  emb_dim)
        self.proj    = nn.Sequential(          # projection head (discarded after pretraining)
            nn.Linear(emb_dim, emb_dim), nn.ReLU(),
            nn.Linear(emb_dim, emb_dim),
        )

    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))
        x = self.conv2(x, edge_index)
        return x

    def project(self, x, edge_index):
        """Returns projection-head output for contrastive training."""
        return self.proj(self(x, edge_index))


def pretrain_contrastive(data: Data, encoder: nn.Module,
                          epochs: int = 10, batch_size: int = 512,
                          feat_mask: float = 0.2, edge_drop: float = 0.2,
                          device: torch.device = DEVICE,
                          temperature: float = 0.5) -> nn.Module:
    """
    Mini-batch NT-Xent pretraining.
    Each batch: sample B nodes → build induced subgraph → two augmentations → loss.
    """
    encoder   = encoder.to(device)
    optimizer = AdamW(encoder.parameters(), lr=1e-3, weight_decay=1e-4)
    scaler    = torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda'))

    x_full  = data.x.to(device)
    ei_full = data.edge_index.to(device)
    N       = data.num_nodes

    for epoch in range(1, epochs + 1):
        encoder.train()
        perm = torch.randperm(N, device=device)
        epoch_loss = 0.0
        steps = 0

        for start in range(0, N, batch_size):
            batch_nodes = perm[start:start + batch_size]
            B = batch_nodes.size(0)

            # Build induced subgraph (edges where BOTH endpoints are in batch)
            node_mask = torch.zeros(N, dtype=torch.bool, device=device)
            node_mask[batch_nodes] = True
            keep = node_mask[ei_full[0]] & node_mask[ei_full[1]]
            sub_ei = ei_full[:, keep]

            # Remap global → local node indices
            id_map = torch.full((N,), -1, dtype=torch.long, device=device)
            id_map[batch_nodes] = torch.arange(B, device=device)
            sub_ei = id_map[sub_ei]       # now in [0, B)

            x_sub = x_full[batch_nodes]

            # Two augmented views
            x1 = aug_feat_mask(x_sub, feat_mask)
            x2 = aug_feat_mask(x_sub, feat_mask)
            ei1 = aug_edge_dropout(sub_ei, edge_drop)
            ei2 = aug_edge_dropout(sub_ei, edge_drop)

            optimizer.zero_grad()
            with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
                z1 = encoder.project(x1, ei1)
                z2 = encoder.project(x2, ei2)
                loss = nt_xent(z1, z2, temperature)

            scaler.scale(loss).backward()
            scaler.step(optimizer); scaler.update()
            epoch_loss += loss.item(); steps += 1

        print(f'Pretrain epoch {epoch:02d}  avg_loss {epoch_loss/steps:.4f}')

    return encoder


# ─── Run pretraining on Cora ─────────────────────────────────────────────────
set_seed(42)
ds_cora2   = Planetoid(root='./data', name='Cora')
data_cora2 = ds_cora2[0]

pretrain_enc = SAGEPretrainEncoder(
    in_dim=data_cora2.num_node_features, hidden=128, emb_dim=64)
pretrain_enc = pretrain_contrastive(
    data_cora2, pretrain_enc, epochs=10, batch_size=512)
print('Pretraining complete ✓')
print('You can now fine-tune pretrain_enc on a small labelled subset.')


---
## 🖥️ Part 7 · Infrastructure — GPU Monitoring Dashboard

Useful during long training runs to catch:
- **Underutilised GPU** (util < 30%) → increase batch size or model width
- **Memory pressure** (>95% used) → enable gradient checkpointing or reduce batch
- **Thermal throttling** (temp > 85°C) → check cooling / reduce sustained load

Monitoring runs in a background thread so it doesn't block training.


In [ ]:
# ─── GPU monitor (Cell A — run once) ─────────────────────────────────────────
import threading
from datetime import datetime

try:
    import pynvml
    pynvml.nvmlInit()
    _pynvml_ok = True
except Exception:
    _pynvml_ok = False

gpu_log: List[Dict] = []
_stop_monitor = threading.Event()


def _sample_gpu() -> Dict:
    if _pynvml_ok:
        h    = pynvml.nvmlDeviceGetHandleByIndex(0)
        util = pynvml.nvmlDeviceGetUtilizationRates(h).gpu
        mem  = pynvml.nvmlDeviceGetMemoryInfo(h)
        temp = pynvml.nvmlDeviceGetTemperature(h, pynvml.NVML_TEMPERATURE_GPU)
        return {'util': int(util),
                'mem_used': int(mem.used // 1024**2),
                'mem_total': int(mem.total // 1024**2),
                'temp': int(temp)}
    # Fallback: nvidia-smi subprocess
    try:
        raw = os.popen(
            'nvidia-smi --query-gpu=utilization.gpu,memory.used,memory.total,'
            'temperature.gpu --format=csv,noheader,nounits').read().strip()
        util, mem_used, mem_total, temp = [int(v.strip()) for v in raw.split(',')]
        return {'util': util, 'mem_used': mem_used, 'mem_total': mem_total, 'temp': temp}
    except Exception:
        return {'util': 0, 'mem_used': 0, 'mem_total': 0, 'temp': 0}


def _monitor_loop(interval: float):
    while not _stop_monitor.is_set():
        s = _sample_gpu(); s['ts'] = datetime.now().timestamp()
        gpu_log.append(s)
        _stop_monitor.wait(interval)


def start_gpu_monitor(interval: float = 0.5):
    _stop_monitor.clear()
    t = threading.Thread(target=_monitor_loop, args=(interval,), daemon=True)
    t.start()
    print(f'GPU monitor started (pynvml={_pynvml_ok}, interval={interval}s)')
    return t


def stop_gpu_monitor():
    _stop_monitor.set()
    print(f'GPU monitor stopped. Samples: {len(gpu_log)}')


start_gpu_monitor(0.5)


In [ ]:
# ─── Live dashboard (Cell C — run while training) ────────────────────────────
# Works best in a classic Jupyter notebook; in JupyterLab use %matplotlib widget
%matplotlib inline
from IPython.display import display, clear_output

_dash_start = None

def update_dashboard():
    global _dash_start
    fig, axes = plt.subplots(3, 1, figsize=(10, 7), sharex=True)
    ax_u, ax_m, ax_l = axes

    if gpu_log:
        ts   = np.array([g['ts'] for g in gpu_log])
        if _dash_start is None: _dash_start = ts[0]
        t    = ts - _dash_start
        util = np.array([g['util']     for g in gpu_log])
        mem  = np.array([g['mem_used'] for g in gpu_log])
        ax_u.plot(t, util,  color='#2196F3'); ax_u.set_ylim(0, 100)
        ax_m.plot(t, mem,   color='#FF9800')

    if 'train_log' in globals() and train_log:
        ts2  = np.array([g['ts']   for g in train_log])
        loss = np.array([g['loss'] for g in train_log])
        t2   = ts2 - (_dash_start or ts2[0])
        ax_l.plot(t2, loss, color='#4CAF50')

    for ax, ylabel in zip(axes, ['GPU util %', 'GPU mem MB', 'Train loss']):
        ax.set_ylabel(ylabel); ax.grid(True, alpha=0.3)
    ax_l.set_xlabel('Wall time (s)')
    plt.suptitle('Live Training Dashboard', fontsize=13, fontweight='bold')
    plt.tight_layout()
    clear_output(wait=True); display(fig); plt.close(fig)

# Call update_dashboard() periodically during training or after each epoch.
update_dashboard()


In [ ]:
# ─── Post-run analysis (Cell D) ──────────────────────────────────────────────
stop_gpu_monitor()
out_dir = Path('./outputs'); out_dir.mkdir(exist_ok=True)

if gpu_log:
    ts   = np.array([g['ts']       for g in gpu_log]) - gpu_log[0]['ts']
    util = np.array([g['util']     for g in gpu_log])
    mem  = np.array([g['mem_used'] for g in gpu_log])
    temp = np.array([g['temp']     for g in gpu_log])

    print(f'\nGPU summary')
    print(f'  Samples       : {len(gpu_log)}')
    print(f'  Duration      : {ts[-1]:.1f}s')
    print(f'  Peak util     : {util.max()}%    avg {util.mean():.1f}%')
    print(f'  Peak mem      : {mem.max()} MB   avg {mem.mean():.0f} MB')
    print(f'  Max temp      : {temp.max()}°C')

    fig, axes = plt.subplots(3, 1, figsize=(10, 7), sharex=True)
    for ax, arr, label, color in zip(
            axes, [util, mem, temp],
            ['Util (%)', 'Memory (MB)', 'Temp (°C)'],
            ['#2196F3', '#FF9800', '#F44336']):
        ax.plot(ts, arr, color=color, linewidth=0.8)
        ax.set_ylabel(label); ax.grid(True, alpha=0.3)
    axes[-1].set_xlabel('Wall time (s)')
    plt.suptitle('GPU Monitoring — Post-run', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(out_dir / 'gpu_profile.png', dpi=120)
    plt.show()
    print('Saved →', out_dir / 'gpu_profile.png')
else:
    print('No GPU log (CPU runtime or monitor not started).')


---
## 📚 Summary & Next Steps

### What we built

| Part | Concept | Key PyG APIs |
|------|---------|-------------|
| 1 | Graph data model | `Data`, COO `edge_index` |
| 2 | Edge-conditioned conv | `NNConv`, `DataLoader` batching |
| 3 | Edge-aware attention | Custom `MessagePassing`, `LayerNorm` residuals |
| 4 | Real datasets | `Planetoid`, OGB, split helpers |
| 5 | Embeddings + PCA/t-SNE | Penultimate layer extraction |
| 6 | Self-supervised pretraining | NT-Xent, feature mask, edge dropout |
| 7 | GPU monitoring | `pynvml`, threading, live dashboard |

### Recommended next experiments

1. **Graph classification** — add `global_mean_pool` after the GNN, use graph-level labels
2. **Link prediction** — split edges into train/val/test, use `negative_sampling`
3. **Heterogeneous graphs** — `HeteroData`, `to_hetero()` for multi-relational graphs
4. **Scalable training** — `NeighborLoader` (replaces deprecated `NeighborSampler`)
5. **Explainability** — `torch_geometric.explain` (GNNExplainer, CaptumExplainer)
6. **Transfer learning** — pretrain on ogbn-arxiv, fine-tune on small labelled set

### Useful resources

- [PyG docs](https://pytorch-geometric.readthedocs.io/)
- [OGB leaderboard](https://ogb.stanford.edu/docs/leader_nodeprop/)
- [GraphCL paper](https://arxiv.org/abs/2010.13902) — contrastive learning on graphs
- [Benchmarking GNNs](https://arxiv.org/abs/2003.00982) — what actually matters in GNN design
